# Chapter 9 — Assembling MAS with Subagents

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

<!-- provider-note:start -->
## Setting the backbone LLM of your agent

These notebooks run on **Ollama by default**, the setup the book teaches.
If you do nothing, nothing changes: `make_llm()` starts a local Ollama
service when one isn't already running.

To use OpenAI or Anthropic instead:

1. Install the extra: `uv sync --extra openai` or `uv sync --extra anthropic`.
2. Export `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` before launching Jupyter.
3. Set `LLM_PROVIDER=openai` (or `anthropic`), or pass `provider="openai"`
   to `make_llm()`. A key on its own never switches providers, so one
   exported for unrelated work cannot reroute you off the Ollama path.

The switch applies wherever a notebook builds its LLM with `make_llm()`.
A notebook that constructs `OllamaLLM` directly stays on Ollama regardless
of these settings.

If you opted in but forgot to export the key, you will be prompted for it.
That is the safe path on hosted kernels, and it keeps the key out of the
saved notebook. Setting `OLLAMA_API_KEY` alone routes Ollama to Ollama
Cloud.

**Caveat:** the examples are tuned for qwen3. Output on gpt-5 or Claude
will differ from what is printed in the book, and prompt-sensitive
examples (the ch09 evaluator pattern, ch08 supervised trajectories) may
behave noticeably differently.
<!-- provider-note:end -->

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [2]:
from llm_agents_from_scratch.notebook_utils import make_llm

/home/nerdai/Projects/llm-agents-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Examples

### Example 1: Manual Dispatch with UseSubAgentTool

Console logging is enabled below so you can watch the dispatched sub-agent run — its log lines are tagged `[hailstone]`, distinguishing them from the coordinator's own logs in later examples.

In [3]:
import logging

from llm_agents_from_scratch import LLMAgentBuilder
from llm_agents_from_scratch.data_structures import ToolCall
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.subagents import SubAgentSpec, UseSubAgentTool
from llm_agents_from_scratch.tools import SimpleFunctionTool

enable_console_logging(logging.INFO)

llm = make_llm()


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)

spec = SubAgentSpec(
    name="hailstone",
    description="Computes Hailstone sequences using next_number.",
    builder=LLMAgentBuilder(llm=llm, tools=[next_number_tool]),
    max_steps=20,
)
tool = UseSubAgentTool(subagents_registry={spec.name: spec})

tool_call = ToolCall(
    tool_name="from_scratch__use_subagent",
    arguments={
        "name": "hailstone",
        "task": (
            "Compute the full Hailstone sequence for 5 step by step "
            "using next_number, until you reach 1. Report how many "
            "steps it took."
        ),
    },
)
result = await tool(tool_call=tool_call)
print(result.content)

✓ Using Ollama Cloud (deepseek-v4.1-flash)
[hailstone] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the full Hailstone sequence for 5 step by step using next_number, until you reach 1. Report how many steps it took.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the full Hailstone sequence for 5 step by step using next_number, until you reach 1. Report how many steps it took.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_skill
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: <skill_content name="stop-at-one">
# Stop At One

Compute a full sequence from a starting number down to 1
using the `next_num...[TRUNCATED]
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "462a119c-a496-4921-82da-b324da958c4c",
    "tool_name": "next_number",
    "argu...[TRUNCATED]
[hailstone] INFO (llm_agents_fs.TaskH

### Example 2: Automatic Dispatch via the Coordinator

In [4]:
from llm_agents_from_scratch.data_structures import Task

# reuses `spec` from Example 1 -- the same hailstone SubAgentSpec, no
# changes needed. SubAgentSpec.builder builds a fresh LLMAgent per
# dispatch, so it's safe to reuse across coordinators.
hailstone_coordinator = await (
    LLMAgentBuilder().with_llm(llm).with_subagents([spec]).build()
)

task_hailstone = Task(
    instruction="Determine the Hailstone sequence for the number 5.",
)
result_hailstone = await hailstone_coordinator.run(task_hailstone)
print(result_hailstone.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Determine the Hailstone sequence for the number 5.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Determine the Hailstone sequence for the number 5.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_skill
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: <skill_content name="stop-at-one">
# Stop At One

Compute a full sequence from a starting number down to 1
using the `next_num...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "d58f1bfc-baba-47f8-94ef-2552ed744823",
    "tool_name": "from_scratch__use_subag...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Execute the pending tool call: use the 'hailstone' subagent with the given task to compute the full Hailstone sequence for starting number...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Execute the pending t

### Example 3: HITL on the Coordinator

In [5]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.tools.default import SharedConsoleHumanInputTool

# both subagents here do simple, tightly-scoped work, so a small
# local model is plenty
slm = make_llm(role="small")

# two subagents, each with its own SharedConsoleHumanInputTool instance
# -- but the lock is a CLASS-level asyncio.Lock, so both instances
# share it. Dispatched concurrently, their prompts serialize instead
# of racing for stdin.
hailstone_input_tool = SharedConsoleHumanInputTool(agent_name="hailstone")
hailstone_spec = SubAgentSpec(
    name="hailstone",
    description=(
        "Asks the human operator for a starting number, then computes "
        "its Hailstone sequence."
    ),
    builder=LLMAgentBuilder(
        llm=slm,
        tools=[next_number_tool, hailstone_input_tool],
    ),
    max_steps=20,
)

greeter_input_tool = SharedConsoleHumanInputTool(agent_name="greeter")
greeter_spec = SubAgentSpec(
    name="greeter",
    description="Asks the human operator for their name, then greets them.",
    builder=LLMAgentBuilder(llm=slm, tools=[greeter_input_tool]),
    max_steps=5,
)

# the builder's only async work is MCP tool discovery, and this
# coordinator has no MCP providers, so construct the LLMAgent directly.
# The subagent specs above still use builders: SubAgentSpec.builder is a
# recipe, rebuilt fresh on every dispatch.
hitl_coordinator = LLMAgent(
    llm=llm,
    subagents=[hailstone_spec, greeter_spec],
)

task_hitl = Task(
    instruction=(
        "Dispatch both the hailstone and greeter subagents in the same "
        "response so they run concurrently. hailstone should ask the "
        "human for a starting number, then compute its Hailstone "
        "sequence. greeter should ask the human for their name, then "
        "return a friendly greeting. Report both results."
    ),
)
# watch the console: the two prompts complete one at a time, never
# interleaved, thanks to SharedConsoleHumanInputTool's shared lock.
result_hitl = await hitl_coordinator.run(task_hitl)
print(result_hitl.content)

✓ Using Ollama Cloud (deepseek-v4.1-flash)
INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Dispatch both the hailstone and greeter subagents in the same response so they run concurrently. hailstone should ask the human for a...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Dispatch both the hailstone and greeter subagents in the same response so they run concurrently. hailstone should ask the human fo...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
[hailstone] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Ask the human operator for a starting number, then compute the Hailstone sequence for that number and report the full sequence.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Ask the human operator for a starting number, then compute the Hailstone sequence for that number and 

╭───────────────────────────────────────────── Human Input — greeter ─────────────────────────────────────────────╮
│ What's your name?                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

>:

[scripted human] > Andrei
[greeter] INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: Andrei
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "701342af-02fd-4853-bfbb-ebd946daa549",
    "tool_name": "from_scratch__human_inp...[TRUNCATED]
[greeter] INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: Hello, Andrei! It's great to meet you — welcome, and I hope you have a wonderful day!
[greeter] INFO (llm_agents_fs.TaskHandler) :      No new step required.
[greeter] INFO (llm_agents_fs.LLMAgent) :      🏁 Task completed: Hello, Andrei! It's great to meet you — welcome, and I hope you have a wonderful day!
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: Hello, Andrei! It's great to meet you — welcome, and I hope you have a wonderful day!
[hailstone] INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Call the from_scratch__human_input tool with the prompt asking the human operator for a 

╭──────────────────────────────────────────── Human Input — hailstone ────────────────────────────────────────────╮
│ What starting number (a positive integer greater than 1) should I use to compute the Hailstone sequence?        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

>:

[scripted human] > 6
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 6
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "4ca6909c-1804-4827-8299-84bdec93bafa",
    "tool_name": "next_number",
    "argu...[TRUNCATED]
[hailstone] INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Make the next_number tool call with x=6 to get the next value in the Hailstone sequence.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Make the next_number tool call with x=6 to get the next value in the Hailstone sequence.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: next_number
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 3
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "91831ae5-2d58-46a1-bb31-d5e77eb55d63",
    "tool_name": "next_number",
    "argu..

### Example 4: Router/Triage

In [6]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.subagents.recipes import explore_subagent, general_subagent

coordinator = LLMAgent(
    llm=llm,  # the bigger model from Example 1
    subagents=[general_subagent(llm), explore_subagent(llm)],
)

# a lookup task — should route to explore (reads hailstone_known_sequences.json)
task_lookup = Task(
    instruction=(
        "Look up the Hailstone sequence for 6 in "
        "hailstone_known_sequences.json and report it."
    ),
)
result_lookup = await coordinator.run(task_lookup)
print(result_lookup.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Look up the Hailstone sequence for 6 in hailstone_known_sequences.json and report it.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Look up the Hailstone sequence for 6 in hailstone_known_sequences.json and report it.
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
[explore] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Find the file named "hailstone_known_sequences.json" on the filesystem (search likely locations such as the current working directory...[TRUNCATED]
[explore] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Find the file named "hailstone_known_sequences.json" on the filesystem (search likely locations such as the current working direct...[TRUNCATED]
[explore] INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__read_file
[explore] INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__read_file


### Example 5: Parallel Fan-Out

In [7]:
task_fanout = Task(
    instruction=(
        "Dispatch three sub-agent calls in the same response so they run "
        "concurrently: use general to compute the Hailstone sequence for "
        "4, use general to compute it for 8, and use explore to look up "
        "the sequence for 12 in hailstone_known_sequences.json. Report "
        "which of the three starting numbers took the most steps."
    ),
)
# gather() runs the three dispatches concurrently, but wall-clock
# speedup isn't guaranteed: a single local Ollama instance queues
# concurrent requests to the *same* model unless OLLAMA_NUM_PARALLEL is
# raised. Cloud models parallelize for free.
result_fanout = await coordinator.run(task_fanout)
print(result_fanout.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Dispatch three sub-agent calls in the same response so they run concurrently: use general to compute the Hailstone sequence for 4, us...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Dispatch three sub-agent calls in the same response so they run concurrently: use general to compute the Hailstone sequence for 4,...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
[general] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the Hailstone (Collatz) sequence starting from 4, iterating until you reach 1. Count the total number of steps to reach 1 (th...[TRUNCATED]
[general] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the Hailstone (Collatz) sequen

### Example 6: Chaining

In [8]:
coordinator.subagents_registry[spec.name] = spec

chain_task = Task(
    instruction=(
        "Ask explore to read hailstone_known_sequences.json and report "
        "its contents. Compare the recorded step counts yourself and "
        "take the starting number with the most steps. Then ask "
        "hailstone to compute the full Hailstone sequence for double "
        "that number. Report the sequence and whether it took more "
        "steps than the recorded entry."
    ),
)
chain_handler = coordinator.run(chain_task, max_steps=10)
chain_result = await chain_handler
print(chain_result.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Ask explore to read hailstone_known_sequences.json and report its contents. Compare the recorded step counts yourself and take the st...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Ask explore to read hailstone_known_sequences.json and report its contents. Compare the recorded step counts yourself and take the...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
[explore] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Read the file hailstone_known_sequences.json and report its full contents verbatim, including all keys and values. Do not summarize o...[TRUNCATED]
[explore] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Read the file hailstone_known_sequences.json and report its full contents verbatim, including all keys and values. Do not summariz...[TRUNCATED]
[explore] INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: f

`chain_handler.rollout` is what shows this really was a chain. `explore` reports what the file contains, then the coordinator picks the entry with the most steps, doubles it, and writes the result into `hailstone`'s dispatch instruction. That second dispatch could not have been issued until the first one answered, and the number in it appears nowhere in the original task. The coordinator does not just relay a value here, it transforms one.

The doubling also gives us a free correctness check. Any even number `2n` steps straight down to `n`, so a doubled starting number always takes exactly one more step than the original.

In [9]:
print(chain_handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'Ask explore to read hailstone_known_sequences.json and report its contents. Compare the recorded step counts yourself and take the starting number with the most steps. Then ask hailstone to compute the full Hailstone sequence for double that number. Report the sequence and whether it took more steps than the recorded entry.'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "deecd2a9-9650-451f-bbfe-516b1505a1d1",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "explore",
        "task": "Read the file hailstone_known_sequences.json and report its full contents verbatim, including all keys and values. Do not summarize or omit anything."
    }
}.

🔧 tool: {
    "tool_call_id": "deecd2a9-9650-451f-bbfe-516b1505a1d1",
    "content": "Here is the full contents of `hailstone_known_sequences.json`, verbatim:\n\n```json\n{\n  \"6\": {\"sequence\": [6, 3, 10, 5, 16, 8, 4, 2, 1